I have made new eval function

Verify that the order of grasps is returned correctly, and that it achieves the same result as the labels...


In [ ]:
import numpy as np
from graspnetAPI import GraspNetEval, Grasp, GraspGroup
import open3d as o3d
import cv2

import matplotlib.pyplot as plt
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
g = GraspNetEval(root = '/home/bam/graspnetAPI/graspnet', camera = 'realsense', split = 'train')
g.checkDataCompleteness()


Frusterating! got a corrupted file it seems like...
BadZipFile: Bad CRC-32 for file 'offsets.npy'

Scene 50... skip for now

Ok I think it may be caused by Jupyter and reloading, etc...

To change:

1 - Stop using juypter for now
2 - Redownload data, and keep two copies in case it gets corrupted agian!

In [ ]:
scene_id = 10
ann_id = 50
camera = 'kinect'
# camera = 'realsense'

# 0.2 returns fewer grasps then 1.0
_6d_grasp = g.loadGrasp(sceneId = scene_id, annId = ann_id, format = '6d', camera = 'realsense', fric_coef_thresh = 0.2)
print("Number of Grasps: ", len(_6d_grasp))


In [ ]:
_6d_grasp.sort_by_score() # to get best grasps first
# _6d_grasp.sort_by_score(reverse=True) # to get worst grasps first
# print(_6d_grasp[1000])
# selected_grasps = _6d_grasp.random_sample(numGrasp = 10)
# print(selected_grasps[0])
selected_grasps = _6d_grasp[:20]

geometries = []
geometries.append(g.loadScenePointCloud(sceneId = scene_id, annId = ann_id, camera = camera))

# Drawing grasps smalls help see alot of them...
geometries += selected_grasps.to_open3d_geometry_list()
o3d.visualization.draw_geometries(geometries)

In [ ]:
o3d.visualization.draw_geometries(geometries)

In [ ]:
for grasp in selected_grasps:
    # print(grasp)
    print(grasp.score)


In [ ]:
ann_id_list = [ann_id]
grasp_group_list = [selected_grasps]
grasp_list_list, score_list_list, collision_list_list = g.eval_scene_all_grasps(scene_id, ann_id_list, grasp_group_list, vis=True, max_width = 0.1, use_cache = True)


In [ ]:
for score_list in score_list_list:
    for score in score_list:
        print(score)

Ok its worry some for me, that labels that are loaded and are hopeful succesful are failing!!!

I am frusteraed that it doesn't seem to work....

How do they load the grasps in rgbd matters.?

Its this cascading code that....


Inside get_scene_label:

Interesting in scene it loads collisions just from kinect, but in camera it loads it from both indvidually....

Passing them ahead of time should help with caching basically...

```python

def gen_scene_label(graspnet_root, scene_id, dump_folder, camera="both"):



```


Inside get_grid_label:

```python
def get_grid_label(
    scene_id,
    camera,
    ann_id,
    graspnet,
    grasp_labels=None,
    collision_labels=None,
    grasp_thresh=0.1,
):

    grasp_label = get_grasp_label(
        scene_id=scene_id,
        camera=camera,
        ann_id=ann_id,
        graspnet=graspnet,
        grasp_labels=grasp_labels,
        collision_labels=collision_labels,
        grasp_thresh=grasp_thresh,
    )
```
Inside get_grasp_label:

```python

def get_grasp_label(
    scene_id,
    camera,
    ann_id,
    graspnet,
    grasp_labels=None,
    collision_labels=None,
    grasp_thresh=0.1,
):

    grasp = graspnet.loadGrasp(
        sceneId=scene_id,
        annId=ann_id,
        format="6d",
        camera=camera,
        grasp_labels=grasp_labels,
        collision_labels=collision_labels,
        fric_coef_thresh=grasp_thresh,
    )
```

In [ ]:
assert len(grasp_group_list) == len(grasp_list_list), f"Length mismatch: {len(grasp_group_list)} vs {len(grasp_list_list)}"
for i, (g_group, g_list) in enumerate(zip(grasp_group_list, grasp_list_list)):
    # g_group is a GraspGroup, g_list is a numpy array
    arr1 = g_group.grasp_group_array
    arr2 = g_list
    assert len(arr1) == len(arr2), f"Sub-list {i} length mismatch: {len(arr1)} vs {len(arr2)}"
    assert np.allclose(arr1, arr2), f"Sub-list {i} arrays are not close!"
    print(f"Sub-list {i} passed verification: {len(arr1)} grasps match.")

print("Verification complete.")